# DD2610 · Structured Flow Autoencoders

In Colab, select **Runtime → Change runtime type → GPU**. The scripts have passed small local checks. Measure full training time, GPU performance and out-of-distribution (OOD) results on the runs you report.

Each teammate should keep a personal notebook copy and set a distinct `PARTICIPANT` in section 1. Agree on the same code revision, data split and seed before running independently and exchanging metrics. Pilot results measure feasibility. Use the agreed formal protocol for comparisons with the paper.

Read `README.md`, `PROJECT_PLAN.md` and `REPRODUCTION_AUDIT.md` in the project directory for the protocols and implementation differences.

## 1. Load the project and install dependencies

For the shared GitHub branch, use `SOURCE = 'github'`. The first run clones the branch and prints its exact commit. Subsequent runs in the same Colab session reuse that checkout. Compare the printed commit with your teammate's before comparing results. If you have the English project ZIP instead, select `SOURCE = 'zip'` and upload it when prompted. This option also works before the branch is published. Both sources fetch the authors' original repository at the version pinned in `scripts/bootstrap.py`.

The working files live in Colab's temporary storage. The second cell mounts your own Google Drive and stores training results there. After a new runtime starts, run both setup cells again, then resume any interrupted run with the same configuration and output folder.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile
import zipfile

SOURCE = 'github'  # Choose 'github' or 'zip'.
REPO_URL = 'https://github.com/NiciiA/StructuredFlowAutoencoder.git'
BRANCH = 'dd2610-colab-english'
content_dir = Path('/content')

if SOURCE == 'github':
    checkout = content_dir / 'dd2610_teammate_checkout'
    if not checkout.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH,
             REPO_URL, str(checkout)], check=True
        )
    origin = subprocess.check_output(
        ['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True
    ).strip()
    active_branch = subprocess.check_output(
        ['git', '-C', str(checkout), 'branch', '--show-current'], text=True
    ).strip()
    if origin != REPO_URL or active_branch != BRANCH:
        raise RuntimeError(f'Unexpected checkout at {checkout}. Use a fresh Colab runtime.')
    revision = subprocess.check_output(
        ['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True
    ).strip()
    print('Project commit:', revision)
    PROJECT = checkout / 'dd2610'
elif SOURCE == 'zip':
    PROJECT = content_dir / 'DD2610_project_en'
    if not PROJECT.exists():
        print('Choose the English DD2610 project ZIP in the file picker below.', flush=True)
        uploaded = files.upload(target_dir=str(content_dir))
        if len(uploaded) != 1:
            raise ValueError('Select exactly one project ZIP.')
        archive = content_dir / next(iter(uploaded))
        with tempfile.TemporaryDirectory(dir=content_dir) as temporary:
            staging = Path(temporary) / 'extracted'
            staging.mkdir()
            with zipfile.ZipFile(archive) as bundle:
                for entry in bundle.infolist():
                    target = (staging / entry.filename).resolve()
                    if target != staging and staging not in target.parents:
                        raise ValueError(f'Unsafe archive entry: {entry.filename}')
                bundle.extractall(staging)
            root = staging if (staging / 'scripts/bootstrap.py').is_file() else staging / 'dd2610'
            if not (root / 'scripts/bootstrap.py').is_file():
                raise ValueError('This ZIP does not contain the DD2610 experiment package.')
            root.rename(PROJECT)
    else:
        print('Reusing the extracted project in this runtime.')
else:
    raise ValueError("SOURCE must be 'github' or 'zip'.")

if not (PROJECT / 'scripts/bootstrap.py').is_file():
    raise FileNotFoundError(f'Missing project code at {PROJECT}. Check the source and branch.')
os.chdir(PROJECT)
print('Installing dependencies and verifying the pinned author code...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.bootstrap'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.verify_freeze'], check=True)
print('Project directory:', PROJECT)

In [ ]:
drive.mount('/content/drive')
PARTICIPANT = 'nico'  # Set your own ID before training: 'dwei' or 'nico'.
RESUME = False  # Set True only to resume an interrupted run with the same settings.
RESULTS = Path('/content/drive/MyDrive/DD2610/results')
RESULTS.mkdir(parents=True, exist_ok=True)
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (select a GPU runtime for training)')
print('Results:', RESULTS)
os.environ['MPLCONFIGDIR'] = '/content/matplotlib_cache'
os.environ['LOKY_MAX_CPU_COUNT'] = '2'

def run(*args):
    return subprocess.run([sys.executable, *map(str,args)], check=True)

## 2. Check the estimator and model interfaces

Verify the expected losses and gradients, six author model branches (including mixture and 5,000-dimensional HVG input), data splits and metrics with known answers. The HVG unit test uses random tensors to exercise the interface. Section 5 runs the experiment on real RNA data.

In [ ]:
run('-m', 'scripts.verify_estimator_identity', '--output', RESULTS / f'{PARTICIPANT}_identity.json')
run('-m', 'unittest', 'discover', '-s', 'tests', '-v')

## 3. Time a short Pinwheel pilot

Train for three epochs on a small subset and evaluate on validation data. Inspect epoch duration, saved files and memory use to set a budget for formal runs. To resume an interrupted pilot, set `RESUME = True` in section 1 and rerun the same command. Set it back to `False` before a new run. Existing results are protected against accidental overwriting.

In [ ]:
pilot_command = ['-m','scripts.run_experiment','--dataset','pinwheel','--method','sfa_k1',
                 '--profile','pilot','--participant',PARTICIPANT,'--seed','0','--epochs','3',
                 '--output',str(RESULTS / 'benchmarks'),'--tag','manual_pilot']
run(*(pilot_command + (['--resume'] if RESUME else [])))

In [ ]:
from IPython.display import display, Image
pilot_dir = RESULTS / 'benchmarks' / PARTICIPANT / 'pinwheel_sfa_k1_pilot_s0_manual_pilot'
display(Image(filename=str(pilot_dir / 'eval_val_steps100.png')))
print((pilot_dir / 'summary.json').read_text())

## 4. Run the authors' models on four types of data

The course feedback asks the group to cover Pinwheel, MNIST, RNA-seq and pendulum, including a mixture model. The runner uses the model and loss classes from the pinned author commit, with an adapted training and evaluation pipeline. Each run records the adaptations in `adaptations.json`.

Choose one `DATASET` per run: `pinwheel`, `mnist`, `mnist_mixture` or `pendulum`. Prepare RNA separately in section 5.

Use `pilot` for smaller datasets and networks. Use `full` for the proposal's data split and presets close to the released model scale. Set the epoch budget explicitly; the cell passes `EPOCHS` for either profile. The full Pinwheel preset uses the released CNF posterior. To use the Gaussian posterior described in the paper, pass a config with `posterior: gaussian` and a nonzero `posterior_freqs`, such as `configs/official_pinwheel_gaussian.json`. Record the actual settings when comparing results with the paper.

In [ ]:
DATASET = 'mnist_mixture'
PROFILE = 'pilot'
EPOCHS = 3
SEED = 0
RUN_ID = f'{PARTICIPANT}_official_{DATASET}_{PROFILE}_s{SEED}_v1'
official_command = ['-m','scripts.run_official','--dataset',DATASET,'--profile',PROFILE,
                    '--participant',PARTICIPANT,'--seed',str(SEED),'--epochs',str(EPOCHS),
                    '--output-root',str(RESULTS / 'official'),'--run-id',RUN_ID]
run(*(official_command + (['--resume'] if RESUME else [])))

## 5. Prepare the Kang RNA data

The [CPA Kang tutorial](https://cpa-tools.readthedocs.io/en/latest/tutorials/Kang.html) provides the source data. Its `X` matrix has already been normalized, log-transformed and restricted to 5,000 highly variable genes (HVGs). Keep these features and use the log-normalized input as supplied.

The downloaded file contains 13,576 cells, 5,000 genes and **eight cell types**, while the paper describes seven types. Keep the observed labels and report the difference. `orig.ident` identifies STIM and CTRL conditions, so donor-based splitting would require separate verified patient IDs. An assessment of leakage from the upstream feature selection would also require all original genes.

The setup creates two datasets:

- `kang_cellholdout.npz` uses a fixed random 80/10/10 cell split. Use it for the primary clustering comparison across all cell types, and describe the adapted split in the report.
- `kang_cpa_splitB.npz` preserves the released loader's `split_B` labels: 11,576 training cells, 1,226 validation cells and 774 OOD cells. The OOD set tests a change of condition for B cells. Report this audit separately from in-distribution multiclass clustering.

In [ ]:
run('-m','pip','install','-q','-r','requirements-rna.txt')
RNA_DIR = Path('/content/drive/MyDrive/DD2610/data')
RNA_DIR.mkdir(parents=True,exist_ok=True)
RAW_RNA = RNA_DIR / 'kang_normalized_hvg.h5ad'
RNA_FILE = RNA_DIR / 'kang_cellholdout.npz'
RNA_RELEASE_FILE = RNA_DIR / 'kang_cpa_splitB.npz'
run('-m','scripts.download_kang','--output',RAW_RNA)
for target, extra in [(RNA_FILE, []), (RNA_RELEASE_FILE, ['--split-key','split_B'])]:
    if not target.exists():
        run('-m','scripts.prepare_official_data','rna','--input',RAW_RNA,'--output',target,
            '--label-key','cell_type','--input-state','lognorm','--keep-features',*extra)
print(RNA_FILE.with_suffix('.json').read_text()[:1800])

In [ ]:
run('-m','scripts.run_official','--dataset','hvg','--profile','pilot','--epochs','3',
    '--participant',PARTICIPANT,'--seed','0','--rna-file',RNA_FILE,
    '--output-root',RESULTS / 'official','--run-id',f'{PARTICIPANT}_hvg_pilot_s0_v1',
    *(['--resume'] if RESUME else []))

## 6. Compare baselines, implementations and estimators

Choose one `METHOD` per run:

- `sfa_k1`, `sfa_k4` and `sfa_cross` compare posterior estimators using the same model, data split, training seed and budget. Keep `alpha=0` for the primary comparison. K4 averages four velocities before squaring the error. The cross-product uses two independent posterior draws and can yield a negative loss.
- `fm`, `vae` and `latent_fm` are baselines. The LatentFM adaptation trains a latent flow after freezing the autoencoder; the paper describes joint fine-tuning in its second stage.
- `paper_only` uses the frozen continuous model generated in a separate implementation session. It sums the loss over coordinates and uses the paper's forward-time convention. The shared runner supplies common data and evaluates with the Heun solver. See `paper_only/PROVENANCE.md` for the original instructions, inputs and limits of the isolation procedure. Keep the implementation frozen for the comparison.

Report parameter counts, elapsed time and protocol differences. Models with the same hidden width can have different parameter counts.

In [ ]:
METHOD = 'sfa_k4'
DATASET = 'pinwheel'  # Set to 'mnist' for MNIST.
PROFILE = 'pilot'    # Use 'full' for the agreed formal data split.
SEED = 0
EPOCHS = 10
TAG = 'estimator_v1'
core_command = ['-m','scripts.run_experiment','--dataset',DATASET,'--method',METHOD,
                '--profile',PROFILE,'--participant',PARTICIPANT,'--seed',str(SEED),
                '--epochs',str(EPOCHS),'--alpha','0','--output',str(RESULTS / 'benchmarks'),'--tag',TAG]
run(*(core_command + (['--resume'] if RESUME else [])))

## 7. Preview or run a batch of experiments

The `pilot` phase covers five author configurations and the primary comparisons. `core` uses the proposal's full data splits. `mnist_ablation` adds the MNIST K1/K4/cross comparison. `breadth` keeps the smaller data and networks, with a longer training schedule; use validation curves to check convergence.

Start with seed 0, measure runtime, then decide whether to add seeds 1 and 2. Each teammate uses a distinct `PARTICIPANT`. Preview the job names before selecting them with `--only`; `--execute` starts training. Missing RNA data marks that job as blocked. The launcher retains completed results. If the experiment list exceeds the proposal's estimate of 25–40 runs, prioritize the four author experiment types and reduce repetitions where appropriate.

In [ ]:
PHASE = 'pilot'
SEEDS = '0'
suite_command = ['-m','scripts.run_suite','--participant',PARTICIPANT,'--phase',PHASE,
                 '--seeds',SEEDS,'--output',str(RESULTS)]
if 'RNA_FILE' in globals() and RNA_FILE.exists():
    suite_command += ['--rna-file',str(RNA_FILE)]
run(*suite_command)  # Preview the plan.

In [ ]:
EXECUTE_SELECTED = False
ONLY = 'pinwheel_sfa_k1_s0,pinwheel_sfa_k4_s0,pinwheel_sfa_cross_s0'
if EXECUTE_SELECTED:
    run(*suite_command,'--only',ONLY,'--execute',*(['--resume'] if RESUME else []))

## 8. Check ODE sensitivity and run the final test

Evaluate 50, 100 and 200 ODE steps on validation data. Heun requires two function evaluations per step. Fix the architecture, epoch budget, alpha and solver settings on validation data before running the test. Report the prespecified seeds.

These cells reuse the training command currently held in `core_command` from section 6 and load its validation-selected `best.pt`. Verify that the dataset, method, profile, seed and tag match the run you intend to evaluate. If validation leads you to choose a different step count, update the final evaluation command accordingly.

In [ ]:
for steps in [50,100,200]:
    run(*core_command,'--eval-only','--ode-steps',str(steps))

In [ ]:
FINAL_TEST = False  # Set True after fixing the protocol.
if FINAL_TEST:
    run(*core_command,'--eval-only','--evaluate-test','--eval-samples','1000','--ode-steps','100')
# For an author run: run(*official_command, '--eval-only', '--eval-split', 'test')

## 9. Evaluate OOD performance on EMNIST

Select an already trained MNIST latent model. Set `MNIST_RUN_DIR` to its actual directory (including its tag), then enable `RUN_OOD`. EMNIST requires an additional download larger than MNIST. Keep the encoder frozen and reserve EMNIST for evaluation.

The runner fits unsupervised k-means on the OOD test representations across 20 classes and reports NMI and ARI. Identify these results as transductive clustering when comparing them with the separate training-set linear probe.

In [ ]:
RUN_OOD = False
MNIST_RUN_DIR = RESULTS / 'benchmarks' / PARTICIPANT / 'mnist_sfa_k1_full_s0_core'
if RUN_OOD:
    run('-m','scripts.evaluate_ood','--run-dir',MNIST_RUN_DIR,'--samples','1000','--ode-steps','100')

## 10. Compare teammates' runs

Put both teammates' result directories under the same accessible parent, or pass two paths to `scripts.compare_runs`. The comparison reads each run's configuration, data manifest, environment and metrics. Keep the checkpoints for later evaluation.

- `per_run.csv` contains the individual measurements.
- `seed_summary.csv` summarizes distinct seeds separately for each participant and protocol. A single seed has no confidence interval.
- `member_comparison.csv` compares participants with matching methods, seeds, data, training configurations, code and evaluation settings. Two members repeating one seed give a reproducibility check for that seed.
- An empty pairing table can indicate different protocols, configurations or source hashes. Check them before combining the results.

In [ ]:
run('-m','scripts.compare_runs',RESULTS,'--output',RESULTS / 'comparison')
import pandas as pd
display(pd.read_csv(RESULTS / 'comparison' / 'seed_summary.csv').head(20))

### Pinwheel validation table
Read the seven saved section 6 runs for one participant, seed and training tag. Set the tag below to the one used in your runs. A blank latent metric means the method has no corresponding encoder.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

summary_seed = 0
summary_tag = 'main_v1'
methods = ['sfa_k1', 'sfa_k4', 'sfa_cross', 'paper_only', 'fm', 'vae', 'latent_fm']
rows = []
missing_methods = []
for method in methods:
    run_dir = RESULTS / 'benchmarks' / PARTICIPANT / f'pinwheel_{method}_full_s{summary_seed}_{summary_tag}'
    evaluation = run_dir / 'eval_val_steps100.json'
    summary = run_dir / 'summary.json'
    if not evaluation.is_file() or not summary.is_file():
        missing_methods.append(method)
        continue
    metrics = json.loads(evaluation.read_text())
    timing = json.loads(summary.read_text())
    rows.append({
        'Method': method,
        'Validation samples': metrics['n_eval'],
        'W1 (lower is better)': metrics['w1_marginal_euclidean_raw'],
        'Angular probe R²': metrics.get('angular_probe_r2'),
        'MSE increase after shuffling z': metrics.get('shuffle_mse_gap'),
        'Training time (minutes)': timing['training_seconds'] / 60,
    })

print(f'Pinwheel validation results (seed {summary_seed}, tag {summary_tag})')
if rows:
    display(pd.DataFrame(rows).set_index('Method').round(3))
if missing_methods:
    print('No matching results for:', ', '.join(missing_methods))


## 11. Retain evidence for the report

Keep the code version, the LLM prompt and frozen file hashes, environment, split hashes, complete configurations, seeds, failure logs, validation checkpoint-selection procedure, parameter counts, training time and memory. Record differences between the paper, the authors' implementation and each experiment. Consult Canvas for the current deadline and presentation format; the supplied course PDF links to those details.

See `PROJECT_PLAN.md` for the four author experiment types, competitive baselines, comparison of independent implementations, estimator analysis, priorities and division of work. Mark unfinished runs as pending and report measured results only.